### Exercise 5

In [ ]:
from collections import defaultdict

In [ ]:
def build_model_from_flight_data(path, goal=(14,54), one_indexed=True):
    """
    Returns:
      P_hat: dict mapping (s,a) -> dict of s' -> prob
      R_hat: dict mapping (s,a,s') -> expected reward (here typically 0 or 1)
      R_sa : dict mapping (s,a) -> expected reward
      N_sa : counts for (s,a)
      terminals: set of terminal states (goal only here)
    """
    # Counts
    N_sas = defaultdict(int)     # (s,a,s') -> count
    N_sa  = defaultdict(int)     # (s,a) -> count
    sumR_sas = defaultdict(float) # (s,a,s') -> sum rewards
    sumR_sa  = defaultdict(float) # (s,a) -> sum rewards

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Try to parse robustly: accepts commas, spaces, parentheses
            cleaned = line.replace("(", "").replace(")", "").replace(",", " ")
            parts = cleaned.split()

            i, j, a, r, ip, jp = parts[:6]
            i, j, a, r, ip, jp = int(i), int(j), int(a), float(r), int(ip), int(jp)

            if one_indexed:
                i -= 1; j -= 1; ip -= 1; jp -= 1

            s  = (i, j)
            sp = (ip, jp)

            N_sa[(s,a)] += 1
            N_sas[(s,a,sp)] += 1
            sumR_sa[(s,a)] += r
            sumR_sas[(s,a,sp)] += r

    # Build P_hat
    P_hat = {}
    for (s,a), n in N_sa.items():
        next_dict = {}
        # collect all s' seen for this (s,a)
        for (ss,aa,sp), c in N_sas.items():
            if ss == s and aa == a:
                next_dict[sp] = c / n
        P_hat[(s,a)] = next_dict

    # Reward models
    R_hat = {}
    for (s,a,sp), c in N_sas.items():
        R_hat[(s,a,sp)] = sumR_sas[(s,a,sp)] / c

    R_sa = { (s,a): (sumR_sa[(s,a)] / n) for (s,a), n in N_sa.items() }

    terminals = {goal}
    return P_hat, R_hat, R_sa, N_sa, terminals